In [ ]:
%pip install tabulate

: 

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from pprint import pprint
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tabulate import tabulate
import asyncio
import nest_asyncio
import copy
nest_asyncio.apply()

config = {
    "dataset":{
        "dti":"../../Data/scope_onside_common_v3.parquet",
        "adr":"../../Data/final_rxnorm_meddra_v2.parquet"
    },
    "protein_emb_1":{
        "path":  "../../Data/3. Protein_enbeddings/ESM_embeddings_(t33_650m model).parquet",
        "id_col": "id", 
        "emb_col": "embedding"
    },
    "protein_emb_2":{
        "path": "../../Data/3. Protein_enbeddings/GVP-GNN_protein_embeddings.parquet",
        "id_col": "uniprot_id", 
        "emb_col": "embedding"
    },
    "drug_emb_1":{
        "path": "../../Data/2. Drug_embeddings/EGNN_drug_embeddings_v2.parquet", 
        "id_col": "drug_chembl_id", 
        "emb_col": "embedding"
    },
    "drug_emb_2":{
        "path": "../../Data/2. Drug_embeddings/smiles_embeddings_chemberta.parquet", 
        "id_col": "drug_chembl_id", 
        "emb_col": "embedding"
    }
}

: 

In [ ]:
dti_df = pd.read_parquet(config["dataset"]["dti"])
print(dti_df.info())

# copy selected columns to a new df
# drug_chembl_id as drug_id and target_uniprot_id as protein_id
dti_df = dti_df.rename(columns={"drug_chembl_id": "drug_id", "target_uniprot_id": "protein_id"})

df = dti_df.copy()
df = df[["drug_id", "protein_id", "label", "rxcui"]]


if config["protein_emb_1"]["path"]:
    protein_emb_1_df = pd.read_parquet(config["protein_emb_1"]["path"])
    protein_emb_1_df = protein_emb_1_df.rename(columns={config["protein_emb_1"]["id_col"]: "protein_id"})
    df = df.merge(protein_emb_1_df[["protein_id", config["protein_emb_1"]["emb_col"]]], on="protein_id", how="left")
    df = df.rename(columns={config["protein_emb_1"]["emb_col"]: "prot_emb_1"})

if config["protein_emb_2"]["path"]:
    protein_emb_2_df = pd.read_parquet(config["protein_emb_2"]["path"])
    protein_emb_2_df = protein_emb_2_df.rename(columns={config["protein_emb_2"]["id_col"]: "protein_id"})
    df = df.merge(protein_emb_2_df[["protein_id", config["protein_emb_2"]["emb_col"]]], on="protein_id", how="left")
    df = df.rename(columns={config["protein_emb_2"]["emb_col"]: "prot_emb_2"})

if config["drug_emb_1"]["path"]:
    drug_emb_1_df = pd.read_parquet(config["drug_emb_1"]["path"])
    drug_emb_1_df = drug_emb_1_df.rename(columns={config["drug_emb_1"]["id_col"]: "drug_id"})
    df = df.merge(drug_emb_1_df[["drug_id", config["drug_emb_1"]["emb_col"]]], on="drug_id", how="left")
    df = df.rename(columns={config["drug_emb_1"]["emb_col"]: "drug_emb_1"})

if config["drug_emb_2"]["path"]:
    drug_emb_2_df = pd.read_parquet(config["drug_emb_2"]["path"])
    drug_emb_2_df = drug_emb_2_df.rename(columns={config["drug_emb_2"]["id_col"]: "drug_id"})
    df = df.merge(drug_emb_2_df[["drug_id", config["drug_emb_2"]["emb_col"]]], on="drug_id", how="left")
    df = df.rename(columns={config["drug_emb_2"]["emb_col"]: "drug_emb_2"})



print(df.info())



: 

In [ ]:

class ADRData:
    def __init__(self, id_to_name_dict):
        """
        Initializes with a dictionary of {meddra_id: meddra_name}.
        """
        self.id_to_name = id_to_name_dict
        self.unique_ids = sorted(list(id_to_name_dict.keys()))
        
        self.id_to_idx = {adr_id: i for i, adr_id in enumerate(self.unique_ids)}
        self.idx_to_id = {i: adr_id for i, adr_id in enumerate(self.unique_ids)}
        
        self.vocab_size = len(self.unique_ids)

    def encode(self, adr_list):
        """
        Takes a list of ADR IDs and returns a binary vector (1s and 0s).
        Example: ['10028553', '10003041'] -> [0, 1, 0, 0, 1...]
        """
        vector = np.zeros(self.vocab_size, dtype=np.int8)
        
        for adr_id in adr_list:
            if adr_id in self.id_to_idx:
                idx = self.id_to_idx[adr_id]
                vector[idx] = 1
            else:
                print(f"Warning: ADR ID {adr_id} not in vocabulary.")
                
        return vector

    def decode(self, vector):
        """
        Takes a binary vector and returns a list of human-readable ADR names.
        """
        decoded_names = []
        
        active_indices = np.where(vector == 1)[0]
        
        for idx in active_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            decoded_names.append(name)
            
        return decoded_names

    def decode_indices(self, indices):
        """
        Takes a list or array of indices (e.g., [42, 105, 300]) 
        and returns the corresponding ADR names.
        """
        return [self.id_to_name.get(self.idx_to_id[idx], "Unknown ADR") for idx in indices]

    def decode_top_k(self, confidence_array, k=5):
        """
        Takes the raw probability array from the model, finds the top K 
        highest values, and returns names + their confidence scores.
        """
        # Get indices of the top k probabilities
        top_indices = np.argsort(confidence_array)[-k:][::-1]
        
        results = []
        for idx in top_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            conf = confidence_array[idx]
            results.append({"name": name, "confidence": round(float(conf), 4)})
            
        return results

    def decode_with_threshold(self, confidence_array, threshold=0.5):
        """
        Returns all ADRs that pass a specific confidence threshold.
        Useful for seeing everything the model is "sure" about.
        """
        active_indices = np.where(confidence_array >= threshold)[0]
        
        # Sort them by confidence (highest first)
        active_indices = active_indices[np.argsort(confidence_array[active_indices])[::-1]]
        
        return [self.id_to_name.get(self.idx_to_id[idx], "Unknown ADR") for idx in active_indices]


: 

In [ ]:
adrdf = pd.read_parquet(config['dataset']["adr"])
id_name_dict = dict(zip(adrdf['meddra_id'], adrdf['meddra_name']))
adr_manager = ADRData(id_name_dict)

: 

In [ ]:
drug_to_adr_list = adrdf.groupby('rxnorm_ingredient_id')['meddra_id'].apply(list).to_dict()

def get_encoded_adr(drug_id):
    # Get the list of ADRs for this drug, or an empty list if not found
    adrs = drug_to_adr_list.get(drug_id, [])
    return adr_manager.encode(adrs)

# 2. Map the drug_id (e.g., rxcui) to the encoded vector
# This will create a column where each cell is a numpy array
df['adr'] = df['rxcui'].map(get_encoded_adr)
print(f"Total rows with ADRs: {df['adr'].apply(lambda x: x.sum() > 0).sum()}")

: 

In [ ]:
# print number of unique protein and drug ids

print(df["protein_id"].nunique())
print(df["drug_id"].nunique())

# print number of unique rxcui
print(df["rxcui"].nunique())
print(df.head(1))



: 

In [ ]:
df.info()

: 

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Get all unique protein IDs
unique_proteins = df['protein_id'].unique()

# 2. Split protein IDs (not rows) to ensure no leakage
# We'll reserve 10% of proteins for Test and 10% for Validation
train_prot_ids, temp_prot_ids = train_test_split(
    unique_proteins, 
    test_size=0.20, 
    random_state=42
)

val_prot_ids, test_prot_ids = train_test_split(
    temp_prot_ids, 
    test_size=0.50, 
    random_state=42
)

# 3. Create the dataframes based on these ID splits
train_df = df[df['protein_id'].isin(train_prot_ids)]
val_df = df[df['protein_id'].isin(val_prot_ids)]
test_df = df[df['protein_id'].isin(test_prot_ids)]

# --- Verification & Metrics ---
print(f"--- Final Dataset Sizes ---")
print(f"Train Set: {len(train_df)} rows ({len(train_prot_ids)} proteins)")
print(f"Val Set:   {len(val_df)} rows ({len(val_prot_ids)} proteins) - [Model Selection]")
print(f"Test Set:  {len(test_df)} rows ({len(test_prot_ids)} proteins) - [Cold-Protein Eval]")

# 4. Recalculate Positive Weight for Training
num_neg = (train_df['label'] == 0).sum()
num_pos = (train_df['label'] == 1).sum()

# Avoid division by zero just in case
pos_weight_value = num_neg / num_pos if num_pos > 0 else 1.0

print(f"\nNew Positive Weight: {pos_weight_value:.2f}")
print(f"Positive/Negative Ratio in Train: 1:{num_neg/num_pos:.2f}")

: 

: 

In [81]:

class FusionModule(nn.Module):
    def __init__(self, dim1, dim2, output_dim):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(dim1 + dim2, output_dim),
            nn.LayerNorm(output_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
    def forward(self, e1, e2):
        return self.fusion(torch.cat([e1, e2], dim=1))

class MultiTaskFusionVAE(nn.Module):
    def __init__(self, drug_dims, prot_dims, adr_dim=4817, fused_dim=768, latent_dim=256):
        super().__init__()
        
        # 1. Fusion Layers
        # Uses drug_dims (list/tuple e.g., [1024, 512]) and prot_dims
        self.drug_fusion = FusionModule(drug_dims[0], drug_dims[1], fused_dim)
        self.prot_fusion = FusionModule(prot_dims[0], prot_dims[1], fused_dim)
        
        # 2. Contextual Encoder 
        # Variable input: drug fused_dim + protein fused_dim
        encoder_input_dim = fused_dim * 2 
        self.context_encoder = nn.Sequential(
            nn.Linear(encoder_input_dim, 1024), 
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU()
        )
        
        # 3. VAE Bottleneck
        self.fc_mu = nn.Linear(512, latent_dim)
        self.fc_logvar = nn.Linear(512, latent_dim)
        
        # 4. Multi-task Heads
        self.adr_decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(),
            nn.Linear(512, adr_dim)
        )
        
        self.dti_head = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, d1, d2, p1, p2):
        fused_drug = self.drug_fusion(d1, d2) 
        fused_prot = self.prot_fusion(p1, p2)
        
        # Concat the two fused representations
        context_input = torch.cat([fused_drug, fused_prot], dim=1)
        context = self.context_encoder(context_input)
        
        mu = self.fc_mu(context)
        logvar = self.fc_logvar(context)
        z = self.reparameterize(mu, logvar)
        
        adr_logits = self.adr_decoder(z)
        dti_logits = self.dti_head(z)
        
        return dti_logits, adr_logits, mu, logvar

In [ ]:
class MultiTaskLossWrapper(nn.Module):
    def __init__(self, num_tasks=2):
        super(MultiTaskLossWrapper, self).__init__()
        # These represent the 'noise' or uncertainty of each task
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, loss_dti, loss_adr):
        # Task 1: DTI
        precision1 = torch.exp(-self.log_vars[0])
        loss1 = precision1 * loss_dti + self.log_vars[0]

        # Task 2: ADR
        precision2 = torch.exp(-self.log_vars[1])
        loss2 = precision2 * loss_adr + self.log_vars[1]

        return loss1 + loss2

# Initialize this before the training loop
# loss_balancer = MultiTaskLossWrapper().to(device)
# optimizer.add_param_group({'params': loss_balancer.parameters()})

: 

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, 
    precision_recall_curve, auc, precision_score, recall_score, average_precision_score
)

def evaluate_multitask(model, dataloader, device):
    model.eval()
    
    # Storage for DTI targets and predictions
    dti_true, dti_probs = [], []
    # Storage for ADR targets and predictions
    adr_true, adr_probs = [], []

    with torch.no_grad():
        for d1, d2, p1, p2, labels, adr_targets in dataloader:
            d1, d2, p1, p2 = d1.to(device), d2.to(device), p1.to(device), p2.to(device)
            
            # Forward pass: support both VAE (returns 4) and logistic (returns 2)
            outputs = model(d1, d2, p1, p2)
            if isinstance(outputs, tuple):
                if len(outputs) >= 2:
                    dti_logits, adr_logits = outputs[0], outputs[1]
                else:
                    raise ValueError("Model must return at least (dti_logits, adr_logits)")
            else:
                raise ValueError("Model forward should return a tuple of (dti_logits, adr_logits, ...)")
            
            # Convert to probabilities
            dti_p = torch.sigmoid(dti_logits).cpu().numpy()
            adr_p = torch.sigmoid(adr_logits).cpu().numpy()
            
            dti_true.extend(labels.numpy())
            dti_probs.extend(dti_p)
            
            adr_true.extend(adr_targets.numpy())
            adr_probs.extend(adr_p)

    # --- 1. DTI METRICS ---
    dti_true = np.array(dti_true)
    dti_probs = np.array(dti_probs).flatten()
    dti_preds = (dti_probs > 0.5).astype(int)

    # Calculate Precision-Recall AUC (AUPRC)
    precision_pts, recall_pts, _ = precision_recall_curve(dti_true, dti_probs)
    auprc = auc(recall_pts, precision_pts)

    dti_results = {
        "DTI_Accuracy": accuracy_score(dti_true, dti_preds),
        "DTI_F1": f1_score(dti_true, dti_preds),
        "DTI_Precision": precision_score(dti_true, dti_preds),
        "DTI_Recall": recall_score(dti_true, dti_preds),
        "DTI_AUROC": roc_auc_score(dti_true, dti_probs),
        "DTI_AUPRC": auprc
    }

    # --- 2. ADR METRICS (Micro-averaged across 4,817 labels) ---
    adr_true = np.array(adr_true)
    adr_probs = np.array(adr_probs)
    adr_preds = (adr_probs > 0.5).astype(int)
    
    # For ADRs, we usually report Micro-average because labels are sparse
    adr_results = {
        "ADR_Micro_AUROC": roc_auc_score(adr_true, adr_probs, average='micro'),
        "ADR_Macro_AUROC": roc_auc_score(adr_true, adr_probs, average='macro'),
        "ADR_Weighted_AUROC": roc_auc_score(adr_true, adr_probs, average='weighted'),
        "ADR_Weighted_AUPRC": average_precision_score(adr_true, adr_probs, average='weighted'), # Approx
        "ADR_Micro_AUPRC": average_precision_score(adr_true, adr_probs, average='micro'), # Approx
        "ADR_Macro_AUPRC": average_precision_score(adr_true, adr_probs, average='macro'), # Approx
        "ADR_F1": f1_score(adr_true, adr_preds, average="weighted")
    }

    return {**dti_results, **adr_results}

: 

In [ ]:

class DTIVAE_Dataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Drug Inputs (Fusion)
        d1 = torch.tensor(row['drug_emb_1'], dtype=torch.float)
        d2 = torch.tensor(row['drug_emb_2'], dtype=torch.float)
        
        # Protein Inputs (Fusion)
        p1 = torch.tensor(row['prot_emb_1'], dtype=torch.float)
        p2 = torch.tensor(row['prot_emb_2'], dtype=torch.float)
        
        # Targets
        label = torch.tensor(row['label'], dtype=torch.float)
        adr_vector = torch.tensor(row['adr'], dtype=torch.float)
        
        return d1, d2, p1, p2, label, adr_vector



: 

In [ ]:
train_loader = DataLoader(DTIVAE_Dataset(train_df), batch_size=64, shuffle=True)
val_loader = DataLoader(DTIVAE_Dataset(val_df), batch_size=64, shuffle=False)
test_loader = DataLoader(DTIVAE_Dataset(test_df), batch_size=64, shuffle=False)

: 

In [ ]:

def train_model(model, train_loader, test_loader, val_loader, pos_weight, epochs=50, lr=1e-4, device='cuda', monitor=None):
    model.to(device)
    
    # 1. Initialize the dynamic balancer
    loss_balancer = MultiTaskLossWrapper(num_tasks=2).to(device)
    
    # 2. Setup Optimizer to include balancer parameters
    # This is critical: the balancer needs to be updated by the same optimizer
    optimizer = torch.optim.Adam([
        {'params': model.parameters()},
        {'params': loss_balancer.parameters(), 'lr': lr}
    ], lr=lr, weight_decay=1e-4)
    
    # Scheduler tracks Val DTI AUPRC to know when to slow down
    scheduler = ReduceLROnPlateau(
        optimizer, 
        mode='max', 
        patience=5, 
        factor=0.2,
        min_lr=1e-6
)
    
    # Weighting for the binary DTI task imbalance
    dti_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
    
    best_val_score = -1
    best_state_dict = None
    
    print(f"Starting Training on {device}...")
    
    for epoch in range(epochs):
        model.train()
        train_losses = []
        
        # VAE KL-annealing: Slowly introduce KL loss to keep latent space organized
        beta = min(0.04, epoch * 0.001)
        
        for d1, d2, p1, p2, labels, adr_targets in train_loader:
            d1, d2, p1, p2 = d1.to(device), d2.to(device), p1.to(device), p2.to(device)
            labels, adr_targets = labels.to(device), adr_targets.to(device)
            
            optimizer.zero_grad()
            
            # Forward Pass
            dti_logits, adr_logits, mu, logvar = model(d1, d2, p1, p2)
            
            # Individual Task Losses
            loss_dti = dti_criterion(dti_logits.squeeze(), labels)
            loss_adr = F.binary_cross_entropy_with_logits(adr_logits, adr_targets)
            
            # VAE Loss (KL Divergence)
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            kl_loss /= (adr_targets.size(0) * adr_targets.size(1))
            
            # 3. Apply Dynamic Balancing (Replaces static Alpha)
            balanced_task_loss = loss_balancer(loss_dti, loss_adr)
            total_loss = balanced_task_loss + (beta * kl_loss)
            
            total_loss.backward()
            
            # 4. Gradient Clipping: Prevents spikes that cause oscillations
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_losses.append(total_loss.item())
            
        # --- Validation only (standard practice) ---
        val_metrics = evaluate_multitask(model, val_loader, device)
        score = val_metrics['DTI_AUPRC']
        
        is_best = score > best_val_score
        if is_best:
            best_val_score = score
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), 'best_multitask_model.pth')
            print(f"*** New Best Model Saved (Val DTI AUPRC: {best_val_score:.4f}) ***")
        
        scheduler.step(score)
                
        # Calculate learned weights for the thesis report
        with torch.no_grad():
            w_dti = torch.exp(-loss_balancer.log_vars[0]).item()
            w_adr = torch.exp(-loss_balancer.log_vars[1]).item()
        
        table_data = [
            [
                "Validation", 
                val_metrics['DTI_AUROC'], 
                val_metrics['DTI_AUPRC'], 
                val_metrics['DTI_F1'], 
                val_metrics['ADR_Micro_AUROC'],
                val_metrics['ADR_Weighted_AUROC'],
                val_metrics['ADR_Micro_AUPRC'],
                val_metrics['ADR_Weighted_AUPRC'],
                val_metrics['ADR_F1']
            ]
]
        # Function to format a value to 4 significant figures if it's a number
        def format_sf(x):
            return float(f"{x:.4g}") if isinstance(x, (int, float)) else x

        # Apply transformation to your table_data
        formatted_table = [[format_sf(item) for item in row] for row in table_data]

        headers = ["SET","DTI AUROC", "DTI AUPRC", "DTI F1", "ADR MICRO AUROC","ADR WEIGHTED AUROC", "ADR MICRO AUPRC","ADR WEIGHTED AUPRC", "ADR_F1"]

        if monitor is not None:
            monitor.log_epoch(epoch, headers, formatted_table, best = is_best)
        
        # 3. Print the report
        print(f"\n🚀 Epoch {epoch+1}/{epochs} | Loss: {np.mean(train_losses):.4f}")
        print(tabulate(table_data, headers=headers, tablefmt="fancy_grid", floatfmt=".4f"))
        print(f"Weights: w_dti={w_dti:.4f}, w_adr={w_adr:.4f}\n")
    
    # ---- Final Test Evaluation on Best Val Checkpoint ----
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        test_metrics = evaluate_multitask(model, test_loader, device)
        
        table_data = [[
            "Test Set", 
            test_metrics['DTI_AUROC'], 
            test_metrics['DTI_AUPRC'], 
            test_metrics['DTI_F1'], 
            test_metrics['ADR_Micro_AUROC'],
            test_metrics['ADR_Weighted_AUROC'],
            test_metrics['ADR_Micro_AUPRC'],
            test_metrics['ADR_Weighted_AUPRC'],
            test_metrics['ADR_F1']
        ]]
        formatted_table = [[format_sf(item) for item in row] for row in table_data]
        print("\n📊 Final Test Metrics (Best Val Checkpoint):")
        print(tabulate(table_data, headers=headers, tablefmt="fancy_grid", floatfmt=".4f"))
    
    return model

: 

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

sample_row = train_df.iloc[0]

# Detect Drug Dimensions
d1_dim = len(sample_row['drug_emb_1'])
d2_dim = len(sample_row['drug_emb_2'])

# Detect Protein Dimensions
p1_dim = len(sample_row['prot_emb_1'])
p2_dim = len(sample_row['prot_emb_2'])

# Detect ADR Dimension
num_adrs = adr_manager.vocab_size

print(f"--- Detected Dimensions ---")
print(f"Drug Inputs: {d1_dim}, {d2_dim}")
print(f"Protein Inputs: {p1_dim}, {p2_dim}")
print(f"ADR Output: {num_adrs}")

# --- 2. INITIALIZE MODEL ---
# Dim sizes depend on your specific embeddings (e.g., 768 or 1024)
model = MultiTaskFusionVAE(
    drug_dims=[d1_dim, d2_dim], 
    prot_dims=[p1_dim, p2_dim], 
    adr_dim=num_adrs, 
    fused_dim=1028,
    latent_dim=512  # You can still keep latent_dim as a hyperparameter
).to(device)

# Initialize this before the training loop


: 

In [88]:
import os
import sys
sys.path.append(os.path.abspath(".."))
from training_monitor import TrainingMonitor

In [89]:
# monitor = TrainingMonitor(server_url='http://localhost:8000', model_name='VAEUpdatedTestTrainSet')

# monitor.connect()


# trained_model = train_model(
#     model=model,
#     train_loader=train_loader,
#     test_loader=test_loader,
#     val_loader=val_loader,
#     pos_weight=1.84,
#     epochs=60,
#     lr=1e-3,
#     device=device,
#     monitor=monitor
# )


In [90]:
# monitor.finish()

In [91]:
# torch.save(model.state_dict(), 'modelstableler-3.pth')

In [92]:
# def predict_interaction(model, d1, d2, p1, p2, threshold=0.5, adr_top_k=20):
#     """
#     Inference function to predict binding and return both Top-K ADRs 
#     and the full raw ADR probability vector.
#     """
#     model.eval()
#     device = next(model.parameters()).device  # Automatically detect model device
    
#     with torch.no_grad():
#         # 1. Prepare Inputs
#         d1 = d1.to(device).unsqueeze(0)
#         d2 = d2.to(device).unsqueeze(0)
#         p1 = p1.to(device).unsqueeze(0)
#         p2 = p2.to(device).unsqueeze(0)
        
#         # 2. Forward Pass
#         dti_logits, adr_logits, mu, _ = model(d1, d2, p1, p2)
        
#         # 3. Process DTI (Binding)
#         prob = torch.sigmoid(dti_logits).item()
#         will_bind = prob >= threshold
        
#         # 4. Process ADRs (Probabilities)
#         # We apply sigmoid to get probabilities between 0 and 1
#         adr_probs_tensor = torch.sigmoid(adr_logits).squeeze()
        
#         # Get Top-K for quick summary
#         top_probs, top_indices = torch.topk(adr_probs_tensor, adr_top_k)
        
#         # Convert full array to list for CPU/JSON compatibility
#         full_adr_array = adr_probs_tensor.cpu().tolist()
        
#     return {
#         "binding_probability": round(prob, 4),
#         "prediction": "BINDER" if will_bind else "NON-BINDER",
#         "top_adr_indices": top_indices.tolist(),
#         "adr_confidences": [round(p.item(), 4) for p in top_probs],
#         "full_adr_vector": full_adr_array  # <--- This is your full 4,817 array
#     }

In [93]:
# class DTIInferenceRetriever:
#     def __init__(self, dataframe, drug_id_col='rxcui', prot_id_col='protein_id'):
#         self.df = dataframe
#         self.drug_id_col = drug_id_col
#         self.prot_id_col = prot_id_col

#     def get_embeddings(self, drug_id, prot_id):
#         # 1. Locate the row where both IDs match
#         query = self.df[
#             (self.df[self.drug_id_col] == drug_id) & 
#             (self.df[self.prot_id_col] == prot_id)
#         ]
        
#         if query.empty:
#             raise ValueError(f"Pair {drug_id} - {prot_id} not found in the dataset.")
            
#         row = query.iloc[0]
        
#         # 2. Extract and convert to tensors
#         d1 = torch.tensor(row['drug_emb_1'], dtype=torch.float)
#         d2 = torch.tensor(row['drug_emb_2'], dtype=torch.float)
#         p1 = torch.tensor(row['prot_emb_1'], dtype=torch.float)
#         p2 = torch.tensor(row['prot_emb_2'], dtype=torch.float)
        
#         return d1, d2, p1, p2



In [94]:
# retriever = DTIInferenceRetriever(df)


In [95]:
# # Initialize the instance
# drug_id = "73494"
# protein_id = "P08183"

# d1, d2, p1, p2 = retriever.get_embeddings(drug_id, protein_id)


In [96]:
# result = predict_interaction(model, d1, d2, p1, p2, threshold=0.5)
    
# # 3. Output results
# print(f"--- Analysis for Drug: {drug_id} | Protein: {protein_id} ---")
# print(f"Status: {result['prediction']}")
# print(f"Binding Probability: {result['binding_probability'] * 100}%")

# if result['prediction'] == "BINDER":
#     print(f"Predicted Side Effects (ADR Indices): {result['top_adr_indices']}")
#     print(result['adr_confidences'])
#     print(result['full_adr_vector'])
#     print(adr_manager.decode_indices(result['top_adr_indices']))
#     adrs = adr_manager.decode_with_threshold(np.array(result['full_adr_vector']), threshold=0.5)
#     print(adrs)
#     print(len(adrs))

In [97]:
# test = ['Headache', 'Nausea', 'Gas', 'Nervous', 'Diarrhoea', 'Rash', 'Hallucination', 'Vomiting', 'Hypersensitivity', 'Cardiac disorder', 'UTI', 'Constipation', 'Oedema', 'Eye disorder', 'Chest pain', 'Carbuncle', 'Asthenia', 'Convulsion', 'Erethism', 'Dizziness', 'Thirst', 'Dyspepsia', 'Inappetence', 'Hepatic function disorder', 'Sleeplessness', 'Urticaria', 'Dry mouth', 'Sleepiness', 'Vision blurred', 'Fatigue', 'Hypertension', 'Insomnia', 'Somnolence', 'Measles', 'Influenza', 'Neuroleptic malignant syndrome', 'Queasy', 'Diarrhea', 'Myalgia', 'Depression', 'Discomfort', 'Orthostatic hypotension', 'Abdominal pain', 'Macule', 'Chest discomfort', 'Stomatitis', 'Nervous system disorder', 'Anorexia', 'Aggression', 'Anxiety', 'Itching', 'Palpitations', 'Vertigo', 'Vascular disorder', 'Renal impairment', 'Gastrointestinal disorder', 'Cough', 'Helplessness', 'DIC', 'Dyskinesia', 'Paraesthesia', 'Glaucoma', 'Jaundice', 'Flatulence', 'Hot flush', 'Arthralgia', 'Consciousness disturbed', 'Anaphylaxis', 'Hepatic impairment']

In [98]:
# print(len(test))

**trying different classifcation models**

**Logistic Regression**

In [99]:
class MultiTaskLogisticRegression(nn.Module):
    """
    Simple multitask logistic regression baseline.
    Uses concatenated raw drug/protein embeddings as input and
    predicts both DTI (binary) and ADR (multi-label) in one shot.
    """
    def __init__(self, input_dim, adr_dim):
        super().__init__()
        # Linear heads = logistic regression when combined with BCEWithLogitsLoss
        self.dti_head = nn.Linear(input_dim, 1)
        self.adr_head = nn.Linear(input_dim, adr_dim)

    def forward(self, d1, d2, p1, p2):
        # Concatenate all four embeddings into a single feature vector
        x = torch.cat([d1, d2, p1, p2], dim=1)
        dti_logits = self.dti_head(x)
        adr_logits = self.adr_head(x)
        return dti_logits, adr_logits


In [100]:
def train_logistic_model(model, train_loader, val_loader, test_loader, pos_weight,
                         epochs=50, lr=1e-3, device='cuda', monitor=None):
    """
    Train + validate a multitask logistic regression model using validation during training,
    then evaluate once on the test set with the best validation checkpoint.
    """
    model.to(device)

    # Dynamic task balancer (same as in VAE training)
    loss_balancer = MultiTaskLossWrapper(num_tasks=2).to(device)

    # Optimizer updates both model and balancer parameters
    optimizer = torch.optim.Adam([
        {'params': model.parameters()},
        {'params': loss_balancer.parameters(), 'lr': lr}
    ], lr=lr, weight_decay=1e-4)

    dti_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))

    best_val_score = -1
    best_state_dict = None

    print(f"Starting Logistic Regression Training on {device}...")

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for d1, d2, p1, p2, labels, adr_targets in train_loader:
            d1, d2, p1, p2 = d1.to(device), d2.to(device), p1.to(device), p2.to(device)
            labels, adr_targets = labels.to(device), adr_targets.to(device)

            optimizer.zero_grad()

            # Forward: no VAE here, just linear heads
            dti_logits, adr_logits = model(d1, d2, p1, p2)

            # Task losses
            loss_dti = dti_criterion(dti_logits.squeeze(), labels)
            loss_adr = F.binary_cross_entropy_with_logits(adr_logits, adr_targets)

            # Dynamic task weighting
            total_loss = loss_balancer(loss_dti, loss_adr)

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_losses.append(total_loss.item())

        # --- Validation only per epoch ---
        val_metrics = evaluate_multitask(model, val_loader, device)
        score = val_metrics['DTI_AUPRC']

        is_best = score > best_val_score
        if is_best:
            best_val_score = score
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), 'best_logreg_multitask_model.pth')
            print(f"*** New Best Logistic Model Saved (Val DTI AUPRC: {best_val_score:.4f}) ***")

        # Learned task weights (for reporting)
        with torch.no_grad():
            w_dti = torch.exp(-loss_balancer.log_vars[0]).item()
            w_adr = torch.exp(-loss_balancer.log_vars[1]).item()

        table_data = [
            [
                "Validation", 
                val_metrics['DTI_AUROC'], 
                val_metrics['DTI_AUPRC'], 
                val_metrics['DTI_F1'], 
                val_metrics['ADR_Micro_AUROC'],
                val_metrics['ADR_Weighted_AUROC'],
                val_metrics['ADR_Micro_AUPRC'],
                val_metrics['ADR_Weighted_AUPRC'],
                val_metrics['ADR_F1']
            ]
]

        def format_sf(x):
            return float(f"{x:.4g}") if isinstance(x, (int, float)) else x

        formatted_table = [[format_sf(item) for item in row] for row in table_data]

        headers = ["SET","DTI AUROC", "DTI AUPRC", "DTI F1", 
                   "ADR MICRO AUROC","ADR WEIGHTED AUROC", 
                   "ADR MICRO AUPRC","ADR WEIGHTED AUPRC", "ADR_F1"]

        if monitor is not None:
            monitor.log_epoch(epoch, headers, formatted_table, best=is_best)

        print(f"\nEpoch {epoch+1}/{epochs} | Logistic Loss: {np.mean(train_losses):.4f}")
        print(tabulate(table_data, headers=headers, tablefmt="fancy_grid", floatfmt=".4f"))
        print(f"Weights: w_dti={w_dti:.4f}, w_adr={w_adr:.4f}\n")

    # ---- Final Test Evaluation on Best Val Checkpoint ----
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        test_metrics = evaluate_multitask(model, test_loader, device)

        table_data = [[
            "Test Set", 
            test_metrics['DTI_AUROC'], 
            test_metrics['DTI_AUPRC'], 
            test_metrics['DTI_F1'], 
            test_metrics['ADR_Micro_AUROC'],
            test_metrics['ADR_Weighted_AUROC'],
            test_metrics['ADR_Micro_AUPRC'],
            test_metrics['ADR_Weighted_AUPRC'],
            test_metrics['ADR_F1']
        ]]
        formatted_table = [[format_sf(item) for item in row] for row in table_data]
        print("\n📊 Final Test Metrics (Best Val Checkpoint):")
        print(tabulate(table_data, headers=headers, tablefmt="fancy_grid", floatfmt=".4f"))

    return model

In [101]:
# --- Logistic Regression Baseline: Init + Train ---

# Input dimension = concat(drug_emb_1, drug_emb_2, prot_emb_1, prot_emb_2)
logreg_input_dim = d1_dim + d2_dim + p1_dim + p2_dim
logreg_adr_dim = num_adrs  # same ADR vocab size as VAE

logreg_model = MultiTaskLogisticRegression(
    input_dim=logreg_input_dim,
    adr_dim=logreg_adr_dim,
).to(device)

print(f"Logistic Regression input dim: {logreg_input_dim}")

# Use the same loaders (train_loader, val_loader, test_loader) and
# the same positive weight computed earlier (pos_weight_value).
logreg_trained_model = train_logistic_model(
    model=logreg_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    pos_weight=pos_weight_value,  # or the constant 1.84 you used before
    epochs=50,
    lr=1e-3,
    device=device,
    monitor=None  # or pass your TrainingMonitor instance if you want logging
)

torch.save(logreg_trained_model.state_dict(), 'final_logreg_multitask_model.pth')

Logistic Regression input dim: 2944
Starting Logistic Regression Training on cuda...
*** New Best Logistic Model Saved (Val DTI AUPRC: 0.6542) ***

Epoch 1/50 | Logistic Loss: 0.8177
╒════════════╤═════════════╤═════════════╤══════════╤═══════════════════╤══════════════════════╤═══════════════════╤══════════════════════╤══════════╕
│ SET        │   DTI AUROC │   DTI AUPRC │   DTI F1 │   ADR MICRO AUROC │   ADR WEIGHTED AUROC │   ADR MICRO AUPRC │   ADR WEIGHTED AUPRC │   ADR_F1 │
╞════════════╪═════════════╪═════════════╪══════════╪═══════════════════╪══════════════════════╪═══════════════════╪══════════════════════╪══════════╡
│ Validation │      0.8377 │      0.6542 │   0.6730 │            0.9311 │               0.6819 │            0.4147 │               0.3819 │   0.1767 │
╘════════════╧═════════════╧═════════════╧══════════╧═══════════════════╧══════════════════════╧═══════════════════╧══════════════════════╧══════════╛
Weights: w_dti=1.3756, w_adr=1.1736

*** New Best Logistic Mod

**SVM**

In [ ]:
# --- Linear SVM (SGDClassifier) OvR Baseline on Same Splits ---

from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import numpy as np


def build_feature_matrix(df):
    """Concatenate drug/protein embeddings into a flat feature vector."""
    d1 = np.stack(df['drug_emb_1'].values)
    d2 = np.stack(df['drug_emb_2'].values)
    p1 = np.stack(df['prot_emb_1'].values)
    p2 = np.stack(df['prot_emb_2'].values)
    return np.concatenate([d1, d2, p1, p2], axis=1)


# Reuse the exact same protein-based splits: train_df, val_df, test_df
X_train = build_feature_matrix(train_df)
X_val = build_feature_matrix(val_df)
X_test = build_feature_matrix(test_df)

y_train_dti = train_df['label'].values.astype(int)
y_val_dti = val_df['label'].values.astype(int)
y_test_dti = test_df['label'].values.astype(int)

Y_train_adr = np.stack(train_df['adr'].values)
Y_val_adr = np.stack(val_df['adr'].values)
Y_test_adr = np.stack(test_df['adr'].values)


# 1) DTI: binary Linear SVM

dti_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SGDClassifier(
        loss='hinge',
        penalty='l2',
        alpha=1e-4,
        max_iter=2000,
        n_jobs=-1,
        class_weight='balanced',
        random_state=42,
    )),
])

print("Training Linear SVM (SGDClassifier) for DTI ...")
dti_svm.fit(X_train, y_train_dti)


def evaluate_dti_svm(model, X, y, set_name="Set"):
    scores = model.decision_function(X)
    scores = scores.reshape(-1)
    preds = (scores > 0).astype(int)

    precision_pts, recall_pts, _ = precision_recall_curve(y, scores)
    auprc = auc(recall_pts, precision_pts)

    return {
        "SET": set_name,
        "DTI_AUROC": roc_auc_score(y, scores),
        "DTI_AUPRC": auprc,
        "DTI_F1": f1_score(y, preds),
        "DTI_Accuracy": accuracy_score(y, preds),
        "DTI_Precision": precision_score(y, preds),
        "DTI_Recall": recall_score(y, preds),
    }


dti_val_metrics = evaluate_dti_svm(dti_svm, X_val, y_val_dti, set_name="Val (SVM)")
dti_test_metrics = evaluate_dti_svm(dti_svm, X_test, y_test_dti, set_name="Test (SVM)")


def format_sf(x):
    return float(f"{x:.4g}") if isinstance(x, (int, float)) else x


dti_table = [
    [
        "Validation (SVM)",
        dti_val_metrics["DTI_AUROC"],
        dti_val_metrics["DTI_AUPRC"],
        dti_val_metrics["DTI_F1"],
    ],
    [
        "Test (SVM)",
        dti_test_metrics["DTI_AUROC"],
        dti_test_metrics["DTI_AUPRC"],
        dti_test_metrics["DTI_F1"],
    ],
]

dti_table_fmt = [[format_sf(item) for item in row] for row in dti_table]
dti_headers = ["SET", "DTI AUROC", "DTI AUPRC", "DTI F1"]

print("\n📊 Linear SVM DTI performance (same split):")
print(tabulate(dti_table_fmt, headers=dti_headers, tablefmt="fancy_grid", floatfmt=".4f"))


# 2) ADR: multi-label OvR Linear SVM

adr_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', OneVsRestClassifier(
        SGDClassifier(
            loss='hinge',
            penalty='l2',
            alpha=1e-4,
            max_iter=2000,
            n_jobs=-1,
            random_state=42,
        )
    )),
])

print("\nTraining Linear SVM (SGDClassifier, OvR) for ADR ...")
adr_svm.fit(X_train, Y_train_adr)


def evaluate_adr_svm(model, X, Y, set_name="Set"):
    scores = model.decision_function(X)
    # decision_function from OneVsRestClassifier can be list; convert to array
    if isinstance(scores, list):
        scores = np.vstack(scores).T
    preds = (scores > 0).astype(int)

    return {
        "SET": set_name,
        "ADR_Micro_AUROC": roc_auc_score(Y, scores, average='micro'),
        "ADR_Weighted_AUROC": roc_auc_score(Y, scores, average='weighted'),
        "ADR_Micro_AUPRC": average_precision_score(Y, scores, average='micro'),
        "ADR_Weighted_AUPRC": average_precision_score(Y, scores, average='weighted'),
        "ADR_F1": f1_score(Y, preds, average="weighted"),
    }


adr_val_metrics = evaluate_adr_svm(adr_svm, X_val, Y_val_adr, set_name="Val (SVM)")
adr_test_metrics = evaluate_adr_svm(adr_svm, X_test, Y_test_adr, set_name="Test (SVM)")

adr_table = [
    [
        "Validation (SVM)",
        adr_val_metrics["ADR_Micro_AUROC"],
        adr_val_metrics["ADR_Weighted_AUROC"],
        adr_val_metrics["ADR_Micro_AUPRC"],
        adr_val_metrics["ADR_Weighted_AUPRC"],
        adr_val_metrics["ADR_F1"],
    ],
    [
        "Test (SVM)",
        adr_test_metrics["ADR_Micro_AUROC"],
        adr_test_metrics["ADR_Weighted_AUROC"],
        adr_test_metrics["ADR_Micro_AUPRC"],
        adr_test_metrics["ADR_Weighted_AUPRC"],
        adr_test_metrics["ADR_F1"],
    ],
]

adr_headers = [
    "SET",
    "ADR MICRO AUROC",
    "ADR WEIGHTED AUROC",
    "ADR MICRO AUPRC",
    "ADR WEIGHTED AUPRC",
    "ADR F1",
]

adr_table_fmt = [[format_sf(item) for item in row] for row in adr_table]

print("\n📊 Linear SVM ADR performance (OvR, same split):")
print(tabulate(adr_table_fmt, headers=adr_headers, tablefmt="fancy_grid", floatfmt=".4f"))


: 